In [1]:
# Add the following line to enable inline plotting in Jupyter
%matplotlib inline
import matplotlib.pyplot as plt

# Add open_dvm package to path (temporary until we create and install the package)
import sys
import os
sys.path.insert(0, '/Users/dvm/Documents/DvM')

# Import open_dvm modules
import warnings
warnings.filterwarnings('ignore')

from open_dvm.analysis import BDM
from open_dvm.support.FolderStructure import FolderStructure
from open_dvm.visualization.plot import plot_bdm_timecourse

print("open_dvm imported successfully!")

open_dvm imported successfully!


In [15]:
# ============================================
# CHANGE THIS PATH TO YOUR DATA LOCATION
# ============================================
project_folder = '/Users/dvm/Library/CloudStorage/OneDrive-VrijeUniversiteitAmsterdam/projects/openDvM'
os.chdir(project_folder)

# Subject number (1-7 in this dataset)
sj = 2

# Eye-tracking exclusion: remove trials with fixation breaks
# (eye movements > angle_thresh during window_oi)
eye_dict = {
    'use_tracker': True,  # Enable eye-tracking exclusion
    'window_oi': (0, 0.3),  # Window: 0-300 ms post-stimulus
    'angle_thresh': 1,  # Threshold: 1 degree visual angle
    'viewing_dist': 70,  # Viewing distance (cm)
    'screen_res': (1920, 1080),  # Screen resolution (pixels)
    'screen_h': 29,  # Screen height (cm)
    'drift_correct': (-0.2, 0)  # Drift: -200 to 0 ms
}

# Load preprocessed epochs and behavioral data
# (applies eye-tracking exclusion and artifact correction)
df, epochs = FolderStructure().load_processed_epochs(
    sj, 'ses_01_main', 'main', eye_dict
)

# Display what was loaded
print(f'✓ Subject {sj} loaded successfully!')
print(f'  • Epochs: {len(epochs)} trials')
print(f'  • Behavioral data: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'  • Sampling rate: {epochs.info["sfreq"]} Hz')

Reading /Users/dvm/Library/CloudStorage/OneDrive-VrijeUniversiteitAmsterdam/projects/openDvM/eeg/processed/sub_02_ses_01_main-epo.fif ...
    Found the data of interest:
        t =    -699.22 ...    1000.00 ms
        0 CTF compensation matrices available
Adding metadata with 19 columns
2902 matching events found
No baseline correction applied
0 projection items activated
Eye channel is not specified in eyedict, using HEOG as default
4 trials missing eyetracking
data (used eog instead)
Eye exclusion info saved in preprocessing file (session 1)
✓ Subject 2 loaded successfully!
  • Epochs: 2148 trials
  • Behavioral data: 2148 rows × 20 columns
  • Sampling rate: 512.0 Hz


In [16]:
#distractor image
bdm_o = BDM(sj,epochs,df,'target_loc',baseline = (-0.2,0), 
nr_folds = 10, elec_oi = 'all', 
data_type = 'raw',downsample=128)
 
output = bdm_o.classify(cnds = dict(block_type = ['main']),
    window_oi = (-0.2,0.5), labels_oi = 'all', 
    bdm_name = 'target', GAT = False,
    excl_factor =dict(distractor_presence= ['present']))

Dropped 1340 trials after specifying excl_factor
NOTE DROPPING IS DONE IN PLACE. PLEASE REREAD DATA IF THAT CONDITION IS NECESSARY AGAIN
Applying baseline correction (mode: mean)
downsampling data

You are decoding main. The nr of trials used for
folding is set to 50

The difference between the highest and the lowest
number of observations per class is 415
Fold 10 out of 10 folds in run 1

In [17]:
plt.plot(output[0]['main']['dec_scores'])
plt.savefig('test')